# DRB2 (A) / DRB4 (B) / ds-RNA (C,D) Domain Contact Analysis — per backend, per interactor couple

**Kernel:** `abcfold-drbs-notebook` (`envs/notebook.yaml`) — install once:
```
conda env create -f envs/notebook.yaml
conda activate abcfold-drbs-notebook
python -m ipykernel install --user --name abcfold-drbs-notebook
```

PLIP contacts from `results/rna_ds_drb2_drb4/all_selected_summary.csv`, produced
by `workflows/postprocessing/Snakefile` (stage 3g `run_plip` + 3h `aggregate`).
This is the **DCL4-dropped** re-run of `rna_ds_dcl4_drb2_drb4` — same DRB2 /
DRB4 / dsRNA sequences, no Dicer. Adapted from
`notebooks/rna_ds_dcl4_drb2_drb4_domain_analysis.ipynb`.

**Scope limitation, important:** `configs/rna_ds_drb2_drb4.yaml`'s `plip.chains`
is `[['A'], ['B', 'C', 'D']]` **with `dnareceptor: true`** — receptor = **DRB2
only**, and the dsRNA (C, D) is folded into the *receptor* group alongside it.
PLIP only ever reports receptor-vs-ligand contacts, so this single PLIP pass
can surface exactly one couple: **DRB2 (A) vs DRB4 (B)**. `reschain` is always
`A`, `reschain_lig` is always `B`; **there are zero DRB2–RNA and zero DRB4–RNA
rows** (verified in the "couples" section below). That is a *configuration*
consequence, not a structural finding — do not read the RNA's absence here as
"the RNA doesn't bind". A dedicated RNA-ligand pass is already scaffolded in
`configs/rna_ds_drb2_drb4.yaml`'s `plip_rna_ligands:` block (`dnareceptor:
false`, no `--chains`) but has **not been run**; when it is, add its couples
here.

**Single pose cluster:** `scripts/pose_cluster_anchor.py` put 599/600 models in
one cluster (silhouette k=2, but the "second" cluster is a single outlier
whose PLIP summary is empty). The DRB2 + DRB4 folded domains converge on one
relative pose across all six backends. So — unlike the DCL4 notebook — there
is **no per-cluster stratification** here; the "per cluster" sections are
replaced by a **per-backend** breakdown and a **residue-level interface map**.

**All six backends ran** for this complex (Chai-1 / Protenix / Boltz all
cleared their token limits at ~900 tokens, vs. hard-fail at the DCL4 complex's
~2600). The energy filter below still removes **RosettaFold3** — its minimised
structures are numerically divergent (final energies up to ~1e19 kJ/mol),
exactly the pathology documented in the DCL4 notebook — leaving **AlphaFold3,
Boltz, Chai-1, OpenFold3, Protenix** (five backends) for the analysis.

---

Domain boundaries (1-based inclusive; residue numbering identical to the
sibling project — sequences copied verbatim, see `configs/rna_ds_drb2_drb4.yaml`).
Originally from UniProt PROSITE annotation. **DRB2's dsRBD2/disordered boundary
is the corrected one** (87-188 / 189-434, not the original PROSITE 87-155 /
156-434) — see `notebooks/drb2_drb4_domain_analysis.ipynb`'s fold-upon-binding
investigation (near-universal cross-backend helix formation + low AIUpred
disorder score, both switching sharply at residue 189). DRB4's boundaries are
unrevised.

**DRB2 (chain A) — receptor**

| DRB2 domain | Residues |
|---|---|
| dsRBD1 | 1-70 |
| linker | 71-86 |
| dsRBD2 | 87-188 (was 87-155 under the original PROSITE call) |
| disordered | 189-434 (was 156-434) |

**DRB4 (chain B) — ligand**

| DRB4 domain | Residues |
|---|---|
| dsRBD1 | 4-73 |
| linker | 74-81 |
| dsRBD2 | 82-150 |
| disordered | 151-291 |
| cryoEM_domain | 292-355 |

**ds-RNA (chains C/D)** — no domains, just nucleotide position (C = 57 nt sense
strand, D = 55 nt antisense strand). No contact rows in this PLIP pass (see
scope limitation above).


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

ROOT = Path("..")
RESULTS_DIR = ROOT / "results" / "rna_ds_drb2_drb4"
RECEPTOR_CHAIN, RECEPTOR_NAME = "A", "DRB2"
LIGAND_CHAIN_NAMES = {"B": "DRB4", "C": "RNA(C)", "D": "RNA(D)"}
FIGURES_DIR = RESULTS_DIR / "figures" / "domain_analysis"

TEMPLATE = "plotly_white"
BACKEND_PALETTE = px.colors.qualitative.Set2
ITYPE_PALETTE = px.colors.qualitative.Set1

def out_path(subdir, filename):
    out_dir = FIGURES_DIR / subdir
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / filename

def save_fig(fig, filename, subdir=""):
    out = out_path(subdir, filename)
    fig.write_html(out, include_plotlyjs="cdn")
    print(f"Saved: {out}")
    fig.show()

## Load data


In [2]:
csv_path = RESULTS_DIR / "all_selected_summary.csv"
df = pd.read_csv(csv_path)
df = df.rename(columns={"replica": "cluster", "model": "fname"})
df["cluster"] = df["cluster"].astype(int)

sel = pd.read_csv(RESULTS_DIR / "selected_models.csv")
sel["fname"] = sel["staged_cif"].apply(lambda p: Path(p).stem)
sel = sel[["fname", "cluster", "backend", "seed", "sample_index", "ranking_score", "ptm", "iptm"]]

df = df.merge(sel, on=["fname", "cluster"], how="left", validate="many_to_one")
n_missing_meta = df["backend"].isna().sum()
if n_missing_meta:
    print(f"WARNING: {n_missing_meta} contact rows have no matching selected_models.csv entry")

n_models = df.groupby(["cluster", "fname"]).ngroups
print(f"{csv_path}: {len(df)} contact rows, {df['cluster'].nunique()} pose cluster(s), "
      f"{n_models} model(s), before the energy filter below")
print("\ncontact rows per backend:")
print(df.groupby("backend").size().rename("rows").to_frame())

../results/rna_ds_drb2_drb4/all_selected_summary.csv: 37672 contact rows, 1 pose cluster(s), 546 model(s), before the energy filter below

contact rows per backend:
               rows
backend            
alphafold3     2090
boltz          8811
chai1          9562
openfold3      1532
protenix       1718
rosettafold3  13959


## Filter out numerically-unconverged structures

Robust (median / MAD) modified z-score on each minimised structure's final
ChimeraX energy — same approach as `rna_ds_dcl4_drb2_drb4_domain_analysis.ipynb`
and `drb2_drb4_domain_analysis.ipynb`.

Two deviations from the DCL4 notebook, both because this run has a full
six-backend ensemble to protect:

1. **Whole-backend cut:** if a backend has >50 % of its energy-assessed models
   flagged as outliers, the *entire* backend is dropped (all its models,
   assessed or not). This is what removes RosettaFold3.
2. **Missing `_energy.csv` ⇒ keep:** ~10 % of minimisations completed (a
   `*_fixed.pdb` exists) but ChimeraX did not emit a parseable energy
   trajectory. For a *non-dropped* backend those models are kept rather than
   silently discarded — "not assessed", not "bad".


In [3]:
def read_final_energy(energy_csv_path):
    if not energy_csv_path.exists():
        return np.nan
    last_energy = np.nan
    with open(energy_csv_path) as fh:
        next(fh, None)
        for line in fh:
            parts = line.strip().split(",")
            if len(parts) < 2:
                continue
            try:
                last_energy = float(parts[1])
            except ValueError:
                pass
    return last_energy

energy_rows = []
for pdb_path in sorted(RESULTS_DIR.glob("minimized/*/*/*.pdb")):
    if pdb_path.stem.endswith(("_fixed", "_amber", "_nonprot")):
        continue
    cluster = int(pdb_path.parent.parent.name)
    energy_csv = pdb_path.with_name(pdb_path.stem + "_energy.csv")
    energy_rows.append({"fname": pdb_path.stem, "cluster": cluster,
                        "final_energy": read_final_energy(energy_csv)})

energy_all = pd.DataFrame(energy_rows).merge(
    sel[["fname", "cluster", "backend"]], on=["fname", "cluster"], how="left")
energy_df = energy_all.dropna(subset=["final_energy"]).copy()

MOD_Z_THRESHOLD = 3.5
pooled_median = energy_df["final_energy"].median()
pooled_mad = (energy_df["final_energy"] - pooled_median).abs().median()
energy_df["energy_mod_z"] = 0.6745 * (energy_df["final_energy"] - pooled_median) / pooled_mad
energy_df["energy_ok"] = energy_df["energy_mod_z"].abs() <= MOD_Z_THRESHOLD

print(f"Pooled final-energy median={pooled_median:,.0f} kJ/mol, MAD={pooled_mad:,.0f}")
print(f"\nfinal energy by backend (kJ/mol) — assessed models only:")
print(energy_df.groupby("backend")["final_energy"]
      .agg(["count", "min", "median", "max"]).round(1).to_string())

flag_rate = (1 - energy_df.groupby("backend")["energy_ok"].mean()).rename("flagged_frac")
DROP_BACKENDS = sorted(flag_rate[flag_rate > 0.5].index)
print(f"\nflagged fraction by backend (|mod-z| > {MOD_Z_THRESHOLD}):")
print(flag_rate.round(3).to_string())
print(f"\n=> whole-backend drop (>50% flagged): {DROP_BACKENDS or 'none'}")

Pooled final-energy median=3,838 kJ/mol, MAD=4,803

final energy by backend (kJ/mol) — assessed models only:
              count      min        median           max
backend                                                 
alphafold3       99  -4718.8  9.674500e+03  2.791940e+04
boltz            94  -9108.5  1.107600e+03  1.495010e+04
chai1            90  -6972.7  1.853200e+03  3.335460e+04
openfold3        99  -6267.3  3.326000e+03  1.836620e+04
protenix         91 -11008.4  8.352000e+02  1.620430e+04
rosettafold3     66  -6265.0  2.308130e+09  8.537021e+18

flagged fraction by backend (|mod-z| > 3.5):
backend
alphafold3      0.000
boltz           0.000
chai1           0.011
openfold3       0.000
protenix        0.000
rosettafold3    0.833

=> whole-backend drop (>50% flagged): ['rosettafold3']


In [4]:
# keep = (assessed & energy_ok)  OR  (unassessed)  ... minus every dropped backend
assessed_ok = set(zip(energy_df.loc[energy_df["energy_ok"], "cluster"],
                      energy_df.loc[energy_df["energy_ok"], "fname"]))
unassessed = set(zip(energy_all.loc[energy_all["final_energy"].isna(), "cluster"],
                     energy_all.loc[energy_all["final_energy"].isna(), "fname"]))
keep_pairs = assessed_ok | unassessed

drop_fnames = set(sel.loc[sel["backend"].isin(DROP_BACKENDS), "fname"])
df_keys = pd.MultiIndex.from_arrays([df["cluster"], df["fname"]])
n_models_before = df.groupby(["cluster", "fname"]).ngroups

df = df[df_keys.isin(keep_pairs) & ~df["fname"].isin(drop_fnames)].copy()

n_models_after = df.groupby(["cluster", "fname"]).ngroups
print(f"{n_models_after} / {n_models_before} model(s) kept after the energy filter")
print("\nmodels kept per backend:")
print(df.groupby("backend")["fname"].nunique().rename("n_models").to_frame())

455 / 546 model(s) kept after the energy filter

models kept per backend:
            n_models
backend             
alphafold3        99
boltz             99
chai1             99
openfold3         69
protenix          89


## Domain definitions


In [5]:
DRB2_DOMAINS = [
    ("dsRBD1", 1, 70),
    ("linker", 71, 86),
    ("dsRBD2", 87, 188),        # corrected from 87-155 -- see intro cell
    ("disordered", 189, 434),   # corrected from 156-434
]

DRB4_DOMAINS = [
    ("dsRBD1", 4, 73),
    ("linker", 74, 81),
    ("dsRBD2", 82, 150),
    ("disordered", 151, 291),
    ("cryoEM_domain", 292, 355),
]

def make_domain_mapper(domain_ranges):
    """domain_ranges: list of (label, start, end), inclusive both ends.
    Returns a fn mapping a Series of residue numbers -> domain labels;
    residues outside every range become <NA>."""
    intervals = pd.IntervalIndex.from_tuples(
        [(start, end) for _, start, end in domain_ranges], closed="both")
    labels = [label for label, _, _ in domain_ranges]

    def mapper(resnr_series):
        idx = intervals.get_indexer(resnr_series.astype(float))
        return pd.Series([labels[i] if i != -1 else pd.NA for i in idx],
                         index=resnr_series.index, dtype="object")
    return mapper

DRB2_LABELS = [l for l, _, _ in DRB2_DOMAINS]
DRB4_LABELS = [l for l, _, _ in DRB4_DOMAINS]

drb2_mapper = make_domain_mapper(DRB2_DOMAINS)
LIGAND_DOMAINS = {"B": DRB4_DOMAINS}
LIGAND_LABELS  = {"B": DRB4_LABELS}
LIGAND_MAPPERS = {c: make_domain_mapper(d) for c, d in LIGAND_DOMAINS.items()}

# Receptor = DRB2 = chain A. After fix_pdb (pdb4amber + PDBFixer) the whole
# complex is renumbered continuously across chains rather than restarting each
# chain at 1. Verified directly from a 6-model sample spanning every backend
# (identical in all): chain A (DRB2) 1-434, chain B (DRB4) 435-789,
# chain C (RNA sense) 790-846, chain D (RNA antisense) 847-901. So the
# receptor's own resnr is ALREADY 1-based (no offset); only resnr_lig needs
# the per-chain offset subtracted before domain mapping.
df["drb2_domain"] = drb2_mapper(df["resnr"])

CHAIN_OFFSET = {"B": 434, "C": 789, "D": 846}
df["resnr_lig_raw"] = df["resnr_lig"]
for chain, offset in CHAIN_OFFSET.items():
    mask = df["reschain_lig"] == chain
    df.loc[mask, "resnr_lig"] = df.loc[mask, "resnr_lig"] - offset

for chain, expected_len in [("B", 355), ("C", 57), ("D", 55)]:
    sub = df[df["reschain_lig"] == chain]
    if sub.empty:
        print(f"chain {chain}: no rows in this PLIP pass")
        continue
    bad = sub[~sub["resnr_lig"].between(1, expected_len)]
    if len(bad):
        print(f"WARNING: {len(bad)} chain-{chain} rows outside [1,{expected_len}] "
              f"after offset -- re-verify CHAIN_OFFSET['{chain}'].")
    else:
        print(f"chain {chain}: offset {CHAIN_OFFSET[chain]} verified OK "
              f"(all {len(sub)} rows land in [1,{expected_len}])")

df["ligand_domain"] = pd.NA
for chain, mapper in LIGAND_MAPPERS.items():
    mask = df["reschain_lig"] == chain
    df.loc[mask, "ligand_domain"] = mapper(df.loc[mask, "resnr_lig"])

n_unmapped_r = df["drb2_domain"].isna().sum()
print(f"\nUnmapped DRB2 residues (outside any domain): {n_unmapped_r} "
      f"({100*n_unmapped_r/len(df):.1f}%)")
df[["resnr", "drb2_domain", "reschain_lig", "resnr_lig_raw", "resnr_lig", "ligand_domain"]].head(5)

chain B: offset 434 verified OK (all 23607 rows land in [1,355])
chain C: no rows in this PLIP pass
chain D: no rows in this PLIP pass

Unmapped DRB2 residues (outside any domain): 0 (0.0%)


,resnr,drb2_domain,reschain_lig,resnr_lig_raw,resnr_lig,ligand_domain
0,102,dsRBD2,B,694,260,disordered
1,109,dsRBD2,B,624,190,disordered
2,109,dsRBD2,B,622,188,disordered
3,133,dsRBD2,B,761,327,cryoEM_domain
4,155,dsRBD2,B,781,347,cryoEM_domain


## Interactor couples in this dataset

Every heatmap in this notebook is one **couple**: DRB2 (fixed receptor) vs. one
partner. Three couples are conceivable (DRB2-DRB4, DRB2-RNA(C), DRB2-RNA(D)),
but per the scope limitation in the intro only **DRB2-DRB4** can appear — the
check below makes that explicit so an empty couple later is expected, not a
surprise.


In [6]:
COUPLES = ["B", "C", "D"]   # DRB4, RNA(C), RNA(D)

print("Contact rows per couple (DRB2 vs. partner):")
for c in COUPLES:
    n = (df["reschain_lig"] == c).sum()
    n_models_c = df.loc[df["reschain_lig"] == c, ["cluster", "fname"]].drop_duplicates().shape[0]
    status = "POPULATED" if n else "EMPTY -- no contacts in this PLIP pass (see intro)"
    print(f"  {RECEPTOR_NAME} x {LIGAND_CHAIN_NAMES[c]:8s}: {n:6d} contact rows "
          f"across {n_models_c:3d} model(s) -- {status}")

Contact rows per couple (DRB2 vs. partner):
  DRB2 x DRB4    :  23607 contact rows across 455 model(s) -- POPULATED
  DRB2 x RNA(C)  :      0 contact rows across   0 model(s) -- EMPTY -- no contacts in this PLIP pass (see intro)
  DRB2 x RNA(D)  :      0 contact rows across   0 model(s) -- EMPTY -- no contacts in this PLIP pass (see intro)


In [7]:
def domain_pair_heatmap(data, ligand_chain, title, filename, n_models_norm, subdir="domain_contacts"):
    ligand_name = LIGAND_CHAIN_NAMES[ligand_chain]
    labels = LIGAND_LABELS[ligand_chain]
    sub = data[data["reschain_lig"] == ligand_chain].dropna(subset=["drb2_domain", "ligand_domain"])
    if sub.empty:
        print(f"No {RECEPTOR_NAME}-{ligand_name} contacts in this subset -- skipping '{title}'.")
        return None

    ct = (sub.groupby(["drb2_domain", "ligand_domain"], observed=True).size()
          .unstack(fill_value=0).reindex(index=DRB2_LABELS, columns=labels, fill_value=0))
    rate = ct / n_models_norm if n_models_norm else ct

    fig = go.Figure(go.Heatmap(
        z=rate.values, x=labels, y=DRB2_LABELS, colorscale="YlOrRd",
        text=[[f"{v:.2f}" if v > 0 else "" for v in row] for row in rate.values],
        texttemplate="%{text}", textfont=dict(size=9),
        hovertemplate=f"{RECEPTOR_NAME} domain: %{{y}}<br>{ligand_name} domain: %{{x}}"
                      "<br>%{z:.3f} contacts/model<extra></extra>",
        colorbar=dict(title="Mean<br>contacts/<br>model"),
    ))
    fig.update_layout(
        title=title, xaxis_title=f"{ligand_name} domain", yaxis_title=f"{RECEPTOR_NAME} domain",
        yaxis=dict(autorange="reversed"), template=TEMPLATE,
        width=max(500, len(labels) * 110), height=max(420, len(DRB2_LABELS) * 70),
    )
    save_fig(fig, filename, subdir)
    return ct

def rna_contact_heatmap(data, strand, n_models_norm, title=None, filename=None, subdir="domain_contacts"):
    sub = data[data["reschain_lig"] == strand].dropna(subset=["drb2_domain"])
    if sub.empty:
        print(f"No {RECEPTOR_NAME}-RNA({strand}) contacts in this subset -- skipping.")
        return None
    nt_positions = sorted(sub["resnr_lig"].dropna().unique())
    ct = (sub.groupby(["resnr_lig", "drb2_domain"], observed=True).size()
          .unstack(fill_value=0).reindex(index=nt_positions, columns=DRB2_LABELS, fill_value=0))
    rate = ct / n_models_norm if n_models_norm else ct
    fig = go.Figure(go.Heatmap(
        z=rate.values, x=DRB2_LABELS, y=[str(p) for p in nt_positions], colorscale="YlOrRd",
        hovertemplate=(f"{RECEPTOR_NAME} domain: %{{x}}<br>RNA nt (strand {strand}): %{{y}}"
                       "<br>%{z:.3f} contacts/model<extra></extra>"),
        colorbar=dict(title="Contacts<br>/ model"),
    ))
    fig.update_layout(
        title=title or f"{RECEPTOR_NAME} x RNA strand {strand} contact rate",
        xaxis_title=f"{RECEPTOR_NAME} domain", yaxis_title=f"RNA nt position (strand {strand})",
        yaxis=dict(autorange="reversed", tickfont=dict(size=8)),
        template=TEMPLATE, width=900, height=max(400, len(nt_positions) * 14),
    )
    save_fig(fig, filename or f"drb2_rna_{strand}_heatmap.html", subdir)
    return ct

def couple_heatmap(data, ligand_chain, n_models_norm, title, filename, subdir="domain_contacts"):
    """domain x domain for protein partners (B), nt-position x domain for RNA (C, D)."""
    if ligand_chain == "B":
        return domain_pair_heatmap(data, ligand_chain, title, filename, n_models_norm, subdir)
    return rna_contact_heatmap(data, ligand_chain, n_models_norm, title, filename, subdir)

## One heatmap per couple (all backends pooled)


In [8]:
n_models_total = df.groupby(["cluster", "fname"]).ngroups
ct_all = {}
for c in COUPLES:
    ct_all[c] = couple_heatmap(
        df, c, n_models_total,
        f"{RECEPTOR_NAME} x {LIGAND_CHAIN_NAMES[c]} -- all backends pooled (n={n_models_total} models)",
        f"drb2_{LIGAND_CHAIN_NAMES[c].lower()}_heatmap_pooled.html",
    )

Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/domain_contacts/drb2_drb4_heatmap_pooled.html


No DRB2-RNA(C) contacts in this subset -- skipping.
No DRB2-RNA(D) contacts in this subset -- skipping.


## Interaction-type breakdown (all backends pooled)


In [9]:
print("Interaction type counts:")
display(df["interaction_type"].value_counts().rename("count").to_frame())

print("\nInteraction type by receptor-ligand chain pair:")
display(
    df.groupby(["reschain", "reschain_lig", "interaction_type"], observed=True)
    .size().rename("count").reset_index().sort_values("count", ascending=False).head(20)
)

# contacts/model per interaction type, by backend -- does the ensemble agree?
per_backend_itype = (
    df.groupby(["backend", "interaction_type"], observed=True).size()
      .div(df.groupby("backend")["fname"].nunique(), level="backend")
      .rename("contacts_per_model").reset_index()
)
fig = px.bar(per_backend_itype, x="backend", y="contacts_per_model", color="interaction_type",
             color_discrete_sequence=ITYPE_PALETTE, template=TEMPLATE,
             title=f"{RECEPTOR_NAME} x DRB4 contacts per model, by interaction type and backend")
fig.update_layout(width=760, height=460, xaxis_title="", yaxis_title="contacts / model")
save_fig(fig, "drb2_drb4_interaction_types_by_backend.html")

Interaction type counts:


,count
interaction_type,
hydrogen_bonds,12603
hydrophobic_interactions,8518
salt_bridges,2328
pi-cation_interactions,131
pi-stacking,27



Interaction type by receptor-ligand chain pair:


,reschain,reschain_lig,interaction_type,count
0,A,B,hydrogen_bonds,12603
1,A,B,hydrophobic_interactions,8518
4,A,B,salt_bridges,2328
2,A,B,pi-cation_interactions,131
3,A,B,pi-stacking,27


Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/drb2_drb4_interaction_types_by_backend.html


## Per-backend breakdown

There is only one pose cluster, so instead of the DCL4 notebook's per-cluster
sections this is a **per-backend** comparison: one panel per couple, one
heatmap per surviving backend, shared colour scale within the panel. This is
the cross-architecture agreement check — do independent models place the same
domain-domain contacts?


In [10]:
backends = sorted(df["backend"].dropna().unique())
backend_n_models = df.groupby("backend")["fname"].nunique()
print(f"{len(backends)} backend(s) after the energy filter: " +
      ", ".join(f"{b} (n={backend_n_models[b]})" for b in backends))

5 backend(s) after the energy filter: alphafold3 (n=99), boltz (n=99), chai1 (n=99), openfold3 (n=69), protenix (n=89)


In [11]:
def couple_heatmap_matrix(data, ligand_chain, n_models_norm):
    """Same content as couple_heatmap() but returns (x_labels, y_labels, rate matrix)
    without plotting -- for multi-panel comparisons."""
    if ligand_chain == "B":
        labels = LIGAND_LABELS[ligand_chain]
        sub = data[data["reschain_lig"] == ligand_chain].dropna(subset=["drb2_domain", "ligand_domain"])
        if sub.empty:
            return None
        ct = (sub.groupby(["drb2_domain", "ligand_domain"], observed=True).size()
              .unstack(fill_value=0).reindex(index=DRB2_LABELS, columns=labels, fill_value=0))
        y_labels = DRB2_LABELS
    else:
        sub = data[data["reschain_lig"] == ligand_chain].dropna(subset=["drb2_domain"])
        if sub.empty:
            return None
        nt_positions = sorted(sub["resnr_lig"].dropna().unique())
        ct = (sub.groupby(["resnr_lig", "drb2_domain"], observed=True).size()
              .unstack(fill_value=0).reindex(index=nt_positions, columns=DRB2_LABELS, fill_value=0))
        labels, y_labels = DRB2_LABELS, [str(p) for p in nt_positions]
    return labels, y_labels, (ct / n_models_norm if n_models_norm else ct)

def backend_panel_for_couple(ligand_chain):
    ligand_name = LIGAND_CHAIN_NAMES[ligand_chain]
    per_backend = {}
    for b in backends:
        res = couple_heatmap_matrix(df[df["backend"] == b], ligand_chain, backend_n_models[b])
        if res is not None:
            per_backend[b] = res
    if not per_backend:
        print(f"No {RECEPTOR_NAME}-{ligand_name} contacts for any backend -- skipping panel.")
        return

    present = list(per_backend.keys())
    zmax = max(rate.values.max() for _, _, rate in per_backend.values())
    ncols = min(3, len(present))
    nrows = -(-len(present) // ncols)

    fig = make_subplots(rows=nrows, cols=ncols,
                        subplot_titles=[f"{b} (n={backend_n_models[b]})" for b in present],
                        shared_yaxes=True)
    for i, b in enumerate(present):
        labels, y_labels, rate = per_backend[b]
        row, col = i // ncols + 1, i % ncols + 1
        fig.add_trace(go.Heatmap(
            z=rate.values, x=labels, y=y_labels, colorscale="YlOrRd",
            zmin=0, zmax=zmax, showscale=(b == present[-1]),
            text=[[f"{v:.2f}" if v > 0 else "" for v in r] for r in rate.values],
            texttemplate="%{text}", textfont=dict(size=8),
            hovertemplate=f"{b}<br>{RECEPTOR_NAME}: %{{y}}<br>{ligand_name}: %{{x}}"
                          "<br>%{z:.3f} contacts/model<extra></extra>",
            colorbar=dict(title="Mean<br>contacts/<br>model"),
        ), row=row, col=col)
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(title=f"{RECEPTOR_NAME} x {ligand_name} domain contact pairs by backend",
                      template=TEMPLATE, width=max(760, 380 * ncols),
                      height=max(430, 150 * nrows))
    save_fig(fig, f"drb2_{ligand_name.lower()}_heatmap_by_backend.html", "per_backend")

for c in COUPLES:
    backend_panel_for_couple(c)

Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/per_backend/drb2_drb4_heatmap_by_backend.html


No DRB2-RNA(C) contacts for any backend -- skipping panel.
No DRB2-RNA(D) contacts for any backend -- skipping panel.


## Residue-level interface map

Domain x domain is coarse when the whole ensemble converges on **one**
interface. These panels drop to single-residue resolution: contact rate
(contacts per model) for the busiest DRB2 and DRB4 residues, split by
interaction type, plus the 2-D DRB2-residue x DRB4-residue contact map — the
actual predicted interface footprint, pooled over all five backends.


In [12]:
TOP_N = 25
bc = df[df["reschain_lig"] == "B"].copy()
n_norm = df.groupby(["cluster", "fname"]).ngroups

def residue_rate_bar(data, resnr_col, restype_col, chain_label, filename):
    per_res_total = (data.groupby([resnr_col, restype_col], observed=True).size()
                     .div(n_norm).rename("rate").reset_index()
                     .sort_values("rate", ascending=False).head(TOP_N))
    order = per_res_total[resnr_col].tolist()
    per_res_itype = (data.groupby([resnr_col, restype_col, "interaction_type"], observed=True).size()
                     .div(n_norm).rename("rate").reset_index())
    per_res_itype = per_res_itype[per_res_itype[resnr_col].isin(order)].copy()
    per_res_itype["label"] = (per_res_itype[restype_col].astype(str)
                              + per_res_itype[resnr_col].astype(int).astype(str))
    label_order = [f"{per_res_total.loc[per_res_total[resnr_col] == r, restype_col].iloc[0]}{int(r)}"
                   for r in order]
    fig = px.bar(per_res_itype, x="label", y="rate", color="interaction_type",
                 category_orders={"label": label_order},
                 color_discrete_sequence=ITYPE_PALETTE, template=TEMPLATE,
                 title=f"{chain_label} interface residues -- top {TOP_N} by contact rate "
                       f"(pooled, n={n_norm} models)")
    fig.update_layout(width=950, height=460, xaxis_title=f"{chain_label} residue",
                      yaxis_title="contacts / model", xaxis_tickangle=-45)
    save_fig(fig, filename, "residue_interface")
    return per_res_total

top_drb2 = residue_rate_bar(bc, "resnr", "restype", "DRB2", "drb2_top_residues.html")
top_drb4 = residue_rate_bar(bc, "resnr_lig", "restype_lig", "DRB4", "drb4_top_residues.html")
display(top_drb2.rename(columns={"resnr": "drb2_resnr", "restype": "drb2_restype"}).reset_index(drop=True))
display(top_drb4.rename(columns={"resnr_lig": "drb4_resnr", "restype_lig": "drb4_restype"}).reset_index(drop=True))

Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/residue_interface/drb2_top_residues.html


Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/residue_interface/drb4_top_residues.html


,drb2_resnr,drb2_restype,rate
0,357,ARG,0.714286
1,355,ARG,0.707692
2,342,ARG,0.567033
3,348,PHE,0.503297
4,428,ARG,0.496703
5,110,ARG,0.461538
6,411,ARG,0.448352
7,334,ARG,0.437363
8,105,ARG,0.437363
9,434,ILE,0.430769


,drb4_resnr,drb4_restype,rate
0,323,VAL,0.751648
1,342,PHE,0.745055
2,347,PHE,0.727473
3,346,LYS,0.690110
4,326,ARG,0.681319
5,317,ILE,0.549451
6,283,TRP,0.514286
7,275,ARG,0.494505
8,314,HIS,0.457143
9,319,THR,0.454945


In [13]:
# 2-D interface contact map: DRB2 residue x DRB4 residue, contact rate, pooled.
top_drb2_res = top_drb2["resnr"].tolist()
top_drb4_res = top_drb4["resnr_lig"].tolist()
pair = bc[bc["resnr"].isin(top_drb2_res) & bc["resnr_lig"].isin(top_drb4_res)]
mat = (pair.groupby(["resnr", "resnr_lig"], observed=True).size()
       .div(n_norm).unstack(fill_value=0)
       .reindex(index=sorted(top_drb2_res), columns=sorted(top_drb4_res), fill_value=0))
drb2_tick = {int(r): f"{bc.loc[bc['resnr']==r,'restype'].iloc[0]}{int(r)}" for r in mat.index}
drb4_tick = {int(c): f"{bc.loc[bc['resnr_lig']==c,'restype_lig'].iloc[0]}{int(c)}" for c in mat.columns}
fig = go.Figure(go.Heatmap(
    z=mat.values,
    x=[drb4_tick[c] for c in mat.columns], y=[drb2_tick[r] for r in mat.index],
    colorscale="YlOrRd",
    hovertemplate="DRB2 %{y}<br>DRB4 %{x}<br>%{z:.3f} contacts/model<extra></extra>",
    colorbar=dict(title="contacts<br>/ model")))
fig.update_layout(title=f"DRB2 x DRB4 residue-residue contact map -- top {TOP_N} each, pooled (n={n_norm})",
                  xaxis_title="DRB4 residue", yaxis_title="DRB2 residue",
                  yaxis=dict(autorange="reversed"), template=TEMPLATE, width=900, height=760)
save_fig(fig, "drb2_drb4_residue_contact_map.html", "residue_interface")

Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/residue_interface/drb2_drb4_residue_contact_map.html


## Export contact tables


In [14]:
# pooled + per-backend domain-pair contact-rate matrices, and residue tables
export_dir = out_path("tables", "")
for b in ["_pooled"] + backends:
    data = df if b == "_pooled" else df[df["backend"] == b]
    n_norm_b = data.groupby(["cluster", "fname"]).ngroups
    res = couple_heatmap_matrix(data, "B", n_norm_b)
    if res is None:
        continue
    _, _, rate = res
    tag = "pooled" if b == "_pooled" else b
    p = export_dir / f"drb2_drb4_domain_pair_rate_{tag}.csv"
    rate.to_csv(p)
    print(f"Saved: {p}")

top_drb2.assign(chain="DRB2").rename(columns={"resnr": "resnr", "restype": "restype"}) \
    .to_csv(export_dir / "drb2_top_interface_residues.csv", index=False)
top_drb4.assign(chain="DRB4").rename(columns={"resnr_lig": "resnr", "restype_lig": "restype"}) \
    .to_csv(export_dir / "drb4_top_interface_residues.csv", index=False)
print(f"Saved: {export_dir/'drb2_top_interface_residues.csv'}")
print(f"Saved: {export_dir/'drb4_top_interface_residues.csv'}")

Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb2_drb4_domain_pair_rate_pooled.csv
Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb2_drb4_domain_pair_rate_alphafold3.csv
Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb2_drb4_domain_pair_rate_boltz.csv
Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb2_drb4_domain_pair_rate_chai1.csv
Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb2_drb4_domain_pair_rate_openfold3.csv
Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb2_drb4_domain_pair_rate_protenix.csv
Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb2_top_interface_residues.csv
Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb4_top_interface_residues.csv


---
## Non-MoRF over-folding filter → recomputed per-backend heatmap + symlinks

Same methodology as `notebooks/drb2_drb4_domain_analysis.ipynb`'s
"Non-MoRF over-folding filter" and "ANCHOR2 peak-based MoRF" sections, applied
here on top of the energy filter:

1. **MoRF motifs = ANCHOR2 peak regions.** `scipy.signal.find_peaks` on the
   AIUpred ANCHOR2 (binding-induced-order) track (DRB2's near-saturated track
   is rolling-min baseline-detrended first; DRB4 uses the raw track). DRB2/DRB4
   sequences are identical across all three complexes, so the `drb2_drb4`
   project's AIUpred tracks apply directly by residue number.
2. **Over-folded** = in the **non-MoRF** residues of DRB2 189–434 or DRB4
   151–291, the model forms `> MAX_PCT_HELIX_NONMORF` % helix **or** a single
   contiguous helical run `>= MIN_LONG_HELIX_RUN` (DSSP from
   `results/rna_ds_drb2_drb4/dssp_summary.csv` — plain `<fname>.pdb`, per-chain
   1-based numbering, so **no `CHAIN_OFFSET`** here, unlike the PLIP `df`).
3. Recompute the per-backend DRB2×DRB4 domain heatmap on the doubly-filtered
   set, plus a numeric convergence check, and symlink the survivors (and the
   worst offenders) per backend for ChimeraX.

In [15]:
from scipy.signal import find_peaks, peak_widths
import shutil

# --- filter thresholds ----------------------------------------------------
MAX_PCT_HELIX_NONMORF = 20.0
MIN_LONG_HELIX_RUN    = 12

# --- MoRF motifs via ANCHOR2 peak detection (see drb2_drb4 notebook) -----
_AIU_DIR = ROOT / "data" / "fold_inputs" / "drb2_drb4"
PEAK_CFG = {
    "A": dict(file="AIupred_output_drb2.txt", domain=(189, 434),   # DRB2
              detrend_win=25, height=0.05, prominence=0.10, distance=6, rel_height=0.50),
    "B": dict(file="AIupred_output_drb4.txt", domain=(151, 291),   # DRB4
              detrend_win=0,  height=0.55, prominence=0.15, distance=6, rel_height=0.40),
}

def anchor2_morf_residues(cfg):
    a = pd.read_csv(_AIU_DIR / cfg["file"], sep="\t", comment="#", header=None,
                    names=["pos", "aa", "aiupred_disorder", "anchor2"])
    raw, pos = a["anchor2"].to_numpy(), a["pos"].to_numpy()
    sig = raw
    if cfg["detrend_win"]:
        s = pd.Series(raw)
        base = (s.rolling(cfg["detrend_win"], center=True, min_periods=1).min()
                 .rolling(cfg["detrend_win"], center=True, min_periods=1).mean().to_numpy())
        sig = raw - base
    idx, _ = find_peaks(sig, height=cfg["height"], prominence=cfg["prominence"],
                        distance=cfg["distance"])
    _, _, lips, rips = peak_widths(sig, idx, rel_height=cfg["rel_height"])
    to_res = lambda ff: np.interp(ff, np.arange(len(pos)), pos)
    lo, hi = cfg["domain"]
    res = set()
    for k, li, ri in zip(idx, lips, rips):
        if lo <= pos[k] <= hi:
            res |= set(range(max(int(np.floor(to_res(li))), lo),
                             min(int(np.ceil(to_res(ri))), hi) + 1))
    return res

MORF_RESIDUES = {ch: anchor2_morf_residues(cfg) for ch, cfg in PEAK_CFG.items()}
DISORDERED = {"A": (189, 434), "B": (151, 291)}
for ch, name in [("A", "DRB2"), ("B", "DRB4")]:
    lo, hi = DISORDERED[ch]
    print(f"{name}: {len(MORF_RESIDUES[ch])}/{hi-lo+1} disordered residues are ANCHOR2-peak MoRF "
          f"-> {hi-lo+1-len(MORF_RESIDUES[ch])} non-MoRF residues policed")

# --- per-model non-MoRF helix stats -------------------------------------
dssp_ss = pd.read_csv(RESULTS_DIR / "dssp_summary.csv")
dssp_ss["cluster"] = dssp_ss["cluster"].astype(int)
backend_lookup = sel[["fname", "cluster", "backend"]].drop_duplicates()
_keep = set(zip(df["cluster"], df["fname"]))                       # energy-filtered models
dssp_ss = dssp_ss[[(c, f) in _keep for c, f in zip(dssp_ss["cluster"], dssp_ss["fname"])]]

def _nonmorf_subranges(lo, hi, morf):
    subs, start = [], None
    for r in range(lo, hi + 1):
        if r in morf:
            if start is not None:
                subs.append((start, r - 1)); start = None
        elif start is None:
            start = r
    if start is not None:
        subs.append((start, hi))
    return subs

def _helix_runs(sub):
    s = sub.sort_values(["fname", "cluster", "resnum"]).copy()
    s["is_helix"] = s["ss_type"] == "helix"
    g = s.groupby(["fname", "cluster"], sort=False)["is_helix"]
    s["run_id"] = (s["is_helix"] != g.shift()).groupby([s["fname"], s["cluster"]]).cumsum()
    ho = s[s["is_helix"]]
    return ho.groupby(["fname", "cluster", "run_id"]).agg(length=("resnum", "size")).reset_index()

def nonmorf_stats(chain):
    lo, hi = DISORDERED[chain]
    subs = _nonmorf_subranges(lo, hi, MORF_RESIDUES[chain])
    rn = dssp_ss["resnum"].to_numpy()
    inr = np.zeros(len(dssp_ss), dtype=bool)
    for a, b in subs:
        inr |= (rn >= a) & (rn <= b)
    sub = dssp_ss[(dssp_ss["chain"] == chain) & inr]
    tot = sub.groupby(["fname", "cluster"]).size().rename("n_res")
    hel = sub[sub["ss_type"] == "helix"].groupby(["fname", "cluster"]).size().rename("n_helix")
    out = pd.concat([tot, hel], axis=1)
    out["n_helix"] = out["n_helix"].fillna(0)
    out["pct_helix"] = 100 * out["n_helix"] / out["n_res"]
    mx = {}
    for a, b in subs:
        for (fn, cl), gg in _helix_runs(sub[sub["resnum"].between(a, b)]).groupby(["fname", "cluster"]):
            mx[(fn, cl)] = max(mx.get((fn, cl), 0), int(gg["length"].max()))
    out["max_run"] = [mx.get(k, 0) for k in out.index]
    return out.reset_index()

a2 = nonmorf_stats("A").add_suffix("_drb2").rename(columns={"fname_drb2": "fname", "cluster_drb2": "cluster"})
b4 = nonmorf_stats("B").add_suffix("_drb4").rename(columns={"fname_drb4": "fname", "cluster_drb4": "cluster"})
ss = (df[["cluster", "fname", "backend"]].drop_duplicates()
      .merge(a2, on=["fname", "cluster"], how="left")
      .merge(b4, on=["fname", "cluster"], how="left"))
for c in ["pct_helix_drb2", "max_run_drb2", "n_helix_drb2", "pct_helix_drb4", "max_run_drb4", "n_helix_drb4"]:
    ss[c] = ss[c].fillna(0)
ss["max_run"]    = ss[["max_run_drb2", "max_run_drb4"]].max(axis=1)
ss["worst_pct"]  = ss[["pct_helix_drb2", "pct_helix_drb4"]].max(axis=1)
ss["total_helix"] = ss["n_helix_drb2"] + ss["n_helix_drb4"]
ss["overfolded"] = (ss["worst_pct"] > MAX_PCT_HELIX_NONMORF) | (ss["max_run"] >= MIN_LONG_HELIX_RUN)

print(f"\nNon-MoRF over-folding filter: flag if pct_helix > {MAX_PCT_HELIX_NONMORF} OR "
      f"max_helix_run >= {MIN_LONG_HELIX_RUN} in DRB2 or DRB4 non-MoRF residues\n")
surv = ss.groupby("backend")["overfolded"].agg(n_models="size", n_overfolded="sum")
surv["n_kept"] = surv["n_models"] - surv["n_overfolded"]
surv.loc["TOTAL"] = surv.sum()
display(surv)

print("\nmedian non-MoRF helix stats by backend:")
display(ss.groupby("backend")[["pct_helix_drb2", "max_run_drb2", "pct_helix_drb4", "max_run_drb4"]].median().round(1))

sweep = pd.DataFrame(
    {f"run>={r}": {f"pct>{p}": int((~((ss.worst_pct > p) | (ss.max_run >= r))).sum())
                   for p in (15, 20, 25, 30, 40)}
     for r in (10, 12, 15, 20)})
print(f"\nmodels kept (of {len(ss)}) over a threshold grid:")
display(sweep)

ss_keep = set(zip(ss.loc[~ss["overfolded"], "cluster"], ss.loc[~ss["overfolded"], "fname"]))
_dk = pd.MultiIndex.from_arrays([df["cluster"], df["fname"]])
df_ssf = df[_dk.isin(ss_keep)].copy()
print(f"\ncontacts dataframe after BOTH filters: "
      f"{df_ssf.groupby(['cluster','fname']).ngroups} / {df.groupby(['cluster','fname']).ngroups} models, "
      f"{len(df_ssf)} contact rows")

DRB2: 132/246 disordered residues are ANCHOR2-peak MoRF -> 114 non-MoRF residues policed
DRB4: 67/141 disordered residues are ANCHOR2-peak MoRF -> 74 non-MoRF residues policed



Non-MoRF over-folding filter: flag if pct_helix > 20.0 OR max_helix_run >= 12 in DRB2 or DRB4 non-MoRF residues



,n_models,n_overfolded,n_kept
backend,,,
alphafold3,99,17,82
boltz,99,94,5
chai1,99,49,50
openfold3,69,69,0
protenix,89,32,57
TOTAL,455,261,194



median non-MoRF helix stats by backend:


,pct_helix_drb2,max_run_drb2,pct_helix_drb4,max_run_drb4
backend,,,,
alphafold3,3.5,3.0,8.1,6.0
boltz,29.8,11.0,18.9,5.0
chai1,18.4,9.0,13.5,5.0
openfold3,57.9,14.0,55.4,10.0
protenix,13.2,7.0,9.5,6.0



models kept (of 455) over a threshold grid:


,run>=10,run>=12,run>=15,run>=20
pct>15,146,150,153,153
pct>20,180,194,199,199
pct>25,200,235,246,248
pct>30,220,272,293,302
pct>40,233,291,328,348



contacts dataframe after BOTH filters: 194 / 455 models, 8047 contact rows


### Per-backend DRB2 × DRB4 domain heatmap — energy filter vs. + non-MoRF over-folding filter

Same panel as the "Per-backend breakdown" section, recomputed on the
doubly-filtered `df_ssf`. Backends wiped out by the over-folding filter show
an empty panel with `n=0` — that absence is part of the convergence answer.

In [16]:
def per_backend_domain_panel(data, title, filename, subdir="non_morf_filter"):
    nmod = data.groupby("backend")["fname"].nunique()
    mats = {b: couple_heatmap_matrix(data[data["backend"] == b], "B", int(nmod.get(b, 0)) or 1)
            for b in backends}
    present = [b for b in backends if mats[b] is not None]
    if not present:
        print(f"'{title}': no models survive -- nothing to plot.")
        return {}
    zmax = max(mats[b][2].values.max() for b in present)
    ncols = min(3, len(backends))
    nrows = -(-len(backends) // ncols)
    fig = make_subplots(rows=nrows, cols=ncols,
                        subplot_titles=[f"{b} (n={int(nmod.get(b, 0))})" for b in backends],
                        shared_yaxes=True, shared_xaxes=True,
                        horizontal_spacing=0.05, vertical_spacing=0.13)
    for i, b in enumerate(backends):
        if mats[b] is None:
            continue
        labels, y_labels, rate = mats[b]
        row, col = i // ncols + 1, i % ncols + 1
        fig.add_trace(go.Heatmap(
            z=rate.values, x=labels, y=y_labels, colorscale="YlOrRd",
            zmin=0, zmax=zmax, showscale=(b == present[-1]),
            text=[[f"{v:.2f}" if v > 0 else "" for v in r] for r in rate.values],
            texttemplate="%{text}", textfont=dict(size=8),
            hovertemplate=f"{b}<br>DRB2: %{{y}}<br>DRB4: %{{x}}<br>%{{z:.3f}} contacts/model<extra></extra>",
            colorbar=dict(title="Mean<br>contacts/<br>model")), row=row, col=col)
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(tickangle=45)
    fig.update_layout(title=title, template=TEMPLATE,
                      width=max(1000, 360 * ncols), height=360 * nrows)
    save_fig(fig, filename, subdir)
    return {b: (mats[b][2] if mats[b] is not None else None) for b in backends}

rates_energy = per_backend_domain_panel(
    df, "DRB2 x DRB4 domain contacts by backend -- energy filter only",
    "drb2_drb4_domain_heatmap_by_backend_energyonly.html")
rates_ssf = per_backend_domain_panel(
    df_ssf, "DRB2 x DRB4 domain contacts by backend -- energy + non-MoRF over-folding filter",
    "drb2_drb4_domain_heatmap_by_backend_nonmorf_filtered.html")

# --- numeric convergence: proportional domain-pair map agreement ----------
def _prop(rate_df):
    v = rate_df.values.flatten().astype(float)
    return v / v.sum() if v.sum() else v

def convergence(rates, data):
    rb = {b: r for b, r in rates.items() if r is not None and np.nansum(r.values) > 0}
    bks = sorted(rb)
    C = np.corrcoef(np.vstack([_prop(rb[b]) for b in bks]))
    mean_r = float(np.nanmean(C[np.triu_indices(len(bks), 1)])) if len(bks) > 1 else float("nan")
    tpm = data.groupby(["backend", "cluster", "fname"]).size().groupby("backend").mean()
    cv = float(tpm.std() / tpm.mean())
    return bks, C, mean_r, tpm, cv

for label, rates, data in [("energy filter only", rates_energy, df),
                            ("energy + non-MoRF over-folding filter", rates_ssf, df_ssf)]:
    bks, C, mean_r, tpm, cv = convergence(rates, data)
    print(f"=== {label} ===")
    print(f"backends: {bks}")
    print(f"mean pairwise Pearson r (proportional domain-pair maps): {mean_r:.3f}")
    print(f"contacts/model by backend:\n{tpm.round(1).to_string()}")
    print(f"coefficient of variation of contacts/model across backends: {cv:.3f}")
    display(pd.DataFrame(C, index=bks, columns=bks).round(3))
    print()

Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/non_morf_filter/drb2_drb4_domain_heatmap_by_backend_energyonly.html


Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/non_morf_filter/drb2_drb4_domain_heatmap_by_backend_nonmorf_filtered.html


=== energy filter only ===
backends: ['alphafold3', 'boltz', 'chai1', 'openfold3', 'protenix']
mean pairwise Pearson r (proportional domain-pair maps): 0.668
contacts/model by backend:
backend
alphafold3    21.1
boltz         89.0
chai1         95.5
openfold3     22.2
protenix      19.3
coefficient of variation of contacts/model across backends: 0.793


,alphafold3,boltz,chai1,openfold3,protenix
alphafold3,1.000,0.304,0.176,0.515,0.461
boltz,0.304,1.000,0.879,0.910,0.780
chai1,0.176,0.879,1.000,0.880,0.881
openfold3,0.515,0.910,0.880,1.000,0.898
protenix,0.461,0.780,0.881,0.898,1.000



=== energy + non-MoRF over-folding filter ===
backends: ['alphafold3', 'boltz', 'chai1', 'protenix']
mean pairwise Pearson r (proportional domain-pair maps): 0.536
contacts/model by backend:
backend
alphafold3     19.9
boltz         115.8
chai1          99.0
protenix       15.6
coefficient of variation of contacts/model across backends: 0.835


,alphafold3,boltz,chai1,protenix
alphafold3,1.000,0.258,0.127,0.413
boltz,0.258,1.000,0.912,0.700
chai1,0.127,0.912,1.000,0.805
protenix,0.413,0.700,0.805,1.000


### Symlink survivors (and worst offenders) for ChimeraX

Mirrors `scripts/symlink_overfolded_samples.py` (`--mode clean` /
`--mode overfolded`) but with the ANCHOR2-peak MoRF carve-out used above.
`overfolding_inspection_survivors/<backend>/` = models that **pass** the
filter, cleanest first; `overfolding_inspection/<backend>/` = the N most
over-folded.

In [17]:
def _write_symlinks(mode, out_subdir, top_n=20):
    root = RESULTS_DIR / out_subdir
    if root.exists():
        shutil.rmtree(root)
    root.mkdir(parents=True)
    mroot = (RESULTS_DIR / "minimized").resolve()
    print(f"[{mode}] -> {root}")
    for b in sorted(sel["backend"].dropna().unique()):
        g = ss[ss["backend"] == b].copy()
        n_pool = len(g)
        if mode == "clean":
            g = g[~g["overfolded"]].sort_values(["max_run", "total_helix", "worst_pct"], ascending=True)
        else:
            g = g.sort_values(["max_run", "total_helix"], ascending=False)
        picks = g.head(top_n)
        (root / b).mkdir(parents=True, exist_ok=True)
        n = 0
        for rk, row in enumerate(picks.itertuples(index=False), start=1):
            src = mroot / str(row.cluster) / row.fname / f"{row.fname}.pdb"
            if not src.exists():
                continue
            if mode == "clean":
                nm = f"{rk:02d}_maxhelix{int(row.max_run)}_pct{row.worst_pct:04.1f}_{row.fname}.pdb"
            else:
                nm = f"{rk:02d}_maxhelix{int(row.max_run)}_totalhelix{int(row.total_helix)}_{row.fname}.pdb"
            (root / b / nm).symlink_to(src)
            n += 1
        print(f"  {b:12s}: {n} linked  (eligible {len(g)} / {n_pool})")

_write_symlinks("clean", "overfolding_inspection_survivors")
_write_symlinks("overfolded", "overfolding_inspection")

[clean] -> ../results/rna_ds_drb2_drb4/overfolding_inspection_survivors
  alphafold3  : 20 linked  (eligible 82 / 99)
  boltz       : 5 linked  (eligible 5 / 99)
  chai1       : 20 linked  (eligible 50 / 99)
  openfold3   : 0 linked  (eligible 0 / 69)
  protenix    : 20 linked  (eligible 57 / 89)
  rosettafold3: 0 linked  (eligible 0 / 0)
[overfolded] -> ../results/rna_ds_drb2_drb4/overfolding_inspection
  alphafold3  : 20 linked  (eligible 99 / 99)
  boltz       : 20 linked  (eligible 99 / 99)
  chai1       : 20 linked  (eligible 99 / 99)
  openfold3   : 20 linked  (eligible 69 / 69)
  protenix    : 20 linked  (eligible 89 / 89)
  rosettafold3: 0 linked  (eligible 0 / 0)


### Reading this

**Filter.** ANCHOR2-peak MoRF exempts 132/246 of DRB2's disordered domain and
67/141 of DRB4's; **194 / 455** models pass both filters — AF3 82, Chai-1 50,
Protenix 57, Boltz 5, **OpenFold3 0** (median ~58 % non-MoRF helix, runs to
14). AF3/Protenix keep the tails largely disordered (AF3 ~4/8 % helix,
Protenix ~13/10 %); Boltz sits highest of the survivors (~30 % on DRB2).

**Convergence — different from the binary `drb2_drb4` complex.** Here the
proportional domain-pair maps are already fairly concordant *before* any
over-folding filter (mean pairwise r ≈ 0.67): Boltz / Chai-1 / OpenFold3 /
Protenix agree with each other at r ≈ 0.78–0.91, and **AlphaFold3 is the
lone outlier** (r ≈ 0.18–0.52 to the rest). After the filter (OpenFold3
gone) mean r drops to ≈ 0.54 — Boltz↔Chai-1 stay at 0.91 and Chai-1↔Protenix
at 0.80, while AlphaFold3 stays weakly correlated (0.13–0.41). CV of
contacts/model rises 0.79 → 0.84.

Two axes are in play:
- **magnitude:** AF3 (~20 contacts/model) and Protenix (~16) are light;
  Boltz (~116) and Chai-1 (~99) stay 5–7× heavier *even after* their
  over-folded models are removed — their extra contacts are not the
  helix-over-folding this filter targets (consistent with the coil-compaction
  story in `drb2_drb4_domain_analysis.ipynb`).
- **pattern:** on *which* domain pairs are hot, Protenix leans toward
  Boltz/Chai-1, not AF3. All backends still put the interface in the same
  place — DRB2 disordered × DRB4 disordered / cryoEM_domain (see the pooled
  heatmap section) — but AF3 distributes relatively more weight onto the
  folded dsRBD contacts.

So unlike the binary complex, the filter does **not** leave a single tight
consensus: it leaves a light-contact AF3/Protenix pair and a heavy-contact
Boltz/Chai-1 pair that agree on interface location but not on contact density
or fine pattern. Treat AF3 + Protenix as the conservative consensus; use the
symlinked survivors to eyeball whether Boltz/Chai-1's extra density is real.

**Symlinks written** (`results/rna_ds_drb2_drb4/`):
`overfolding_inspection_survivors/<backend>/` — filter passers, cleanest
first (AF3 20, Chai-1 20, Protenix 20, Boltz 5, OpenFold3 0);
`overfolding_inspection/<backend>/` — 20 worst offenders per backend.

---
## Re-run pose PCA + clustering on the filtered subset

Curiosity check: does dropping the over-folded models change the pose
clustering? This re-implements `scripts/pose_cluster_anchor.py`'s pipeline
(iterative rigid **DRB2** core refinement → per-model Kabsch into that frame →
RMSF-trim the **DRB4** Cα footprint → hierarchical average-linkage clustering
on pairwise partner Cα RMSD with silhouette k-selection → PCA of the pose
feature vector) and runs it twice: on **all energy-filtered** models and on
the **non-MoRF-over-folding survivors** only. Same parameters as the pipeline
(`config.yaml`'s `cluster:` block: core_rmsf 1.5 Å, partner_rmsf 5 Å,
min_frac 0.4, n_iter 8, max_k 12).

**Caveats:** this reads the **minimized** `<fname>.pdb` (post energy-relaxation),
not the raw ABCfold CIFs the pipeline used, and it re-derives the core/trim
per subset — so absolute `pc1`/`pc2` values won't match
`results/rna_ds_drb2_drb4/pose_clusters.csv`. The comparison is
**baseline-vs-filtered within this cell**, not against the pipeline's file.

In [18]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# --- pose_cluster_anchor.py internals, re-implemented on minimized PDBs ----
def _chain_ca(pdb_path, chain):
    xs = []
    with open(pdb_path) as fh:
        for line in fh:
            if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[21] == chain:
                xs.append((float(line[30:38]), float(line[38:46]), float(line[46:54])))
    return np.array(xs, dtype=np.float32)

def _kabsch(Pm, Qm):
    pc, qc = Pm - Pm.mean(0), Qm - Qm.mean(0)
    U, _, Vt = np.linalg.svd(pc.T @ qc)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    Rm = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
    return Rm, Qm.mean(0) - Rm @ Pm.mean(0)

def _refine_core(af, n_iter=8, rmsf_target=1.5, min_frac=0.4):
    ref = af[0]
    core = np.arange(af.shape[1])
    min_sz = max(1, int(min_frac * af.shape[1]))
    for _ in range(n_iter):
        al = np.empty_like(af)
        for m in range(len(af)):
            Rm, t = _kabsch(af[m][core], ref[core])
            al[m] = (Rm @ af[m].T).T + t
        rmsf = np.sqrt(np.mean(np.sum((al - al.mean(0)) ** 2, 2), 0))
        cr = rmsf[core]
        if float(np.median(cr)) <= rmsf_target or len(core) <= min_sz:
            break
        core = core[cr <= np.percentile(cr, 60)]
    return core

def _stable_partner(pa, rmsf_target=5.0, min_frac=0.4):
    rmsf = np.sqrt(np.mean(np.sum((pa - pa.mean(0)) ** 2, 2), 0))
    keep = np.where(rmsf <= rmsf_target)[0]
    min_keep = max(1, int(min_frac * len(rmsf)))
    if len(keep) < min_keep:
        keep = np.argsort(rmsf)[:min_keep]
    return np.sort(keep)

def pose_pipeline(model_pairs, tag):
    """model_pairs: iterable of (cluster, fname). Returns a dict with the
    re-derived PCA + hierarchical clustering for that subset."""
    anchor, partner, meta = [], [], []
    for cl, fn in model_pairs:
        p = RESULTS_DIR / "minimized" / str(cl) / fn / f"{fn}.pdb"
        if not p.exists():
            continue
        a, b = _chain_ca(p, "A"), _chain_ca(p, "B")
        anchor.append(a); partner.append(b); meta.append((cl, fn, len(a), len(b)))
    from collections import Counter
    amode = Counter(m[2] for m in meta).most_common(1)[0][0]
    bmode = Counter(m[3] for m in meta).most_common(1)[0][0]
    keep = [i for i, m in enumerate(meta) if m[2] == amode and m[3] == bmode]
    A = np.stack([anchor[i] for i in keep])
    B = np.stack([partner[i] for i in keep])
    used = [(meta[i][0], meta[i][1]) for i in keep]
    n = len(A)

    core = _refine_core(A)
    ref = A[0]
    PA = np.empty_like(B)
    for m in range(n):
        Rm, t = _kabsch(A[m][core], ref[core])
        PA[m] = (Rm @ B[m].T).T + t
    pcore = _stable_partner(PA)
    X = PA[:, pcore, :].reshape(n, -1)

    d = pdist(X) / np.sqrt(len(pcore))
    Z = linkage(d, method="average")
    D = squareform(d)
    sil = {}
    for k in range(2, 13):
        lab_k = fcluster(Z, t=k, criterion="maxclust")
        if len(set(lab_k)) < 2:
            continue
        sil[k] = float(silhouette_score(D, lab_k, metric="precomputed"))
    best_k = max(sil, key=sil.get) if sil else 1
    labels = fcluster(Z, t=best_k, criterion="maxclust") if best_k > 1 else np.ones(n, int)

    ncomp = min(2, n - 1, X.shape[1])
    pca = PCA(n_components=ncomp, random_state=42)
    pcs = pca.fit_transform(X)

    out = pd.DataFrame(used, columns=["cluster", "fname"])
    out["pose_label"] = labels
    out["pc1"] = pcs[:, 0]
    out["pc2"] = pcs[:, 1] if ncomp > 1 else 0.0
    out = out.merge(sel[["fname", "cluster", "backend"]].drop_duplicates(), on=["fname", "cluster"], how="left")
    print(f"[{tag}] n={n}  DRB2 core={len(core)}  DRB4 trim kept={len(pcore)}  "
          f"best_k={best_k} (silhouette {sil.get(best_k, float('nan')):.3f})  "
          f"PCA var={[round(v,3) for v in pca.explained_variance_ratio_]}  "
          f"sizes={out['pose_label'].value_counts().to_dict()}")
    return {"tag": tag, "df": out, "sil": sil, "var": list(pca.explained_variance_ratio_)}

ef_models   = list(df[["cluster", "fname"]].drop_duplicates().itertuples(index=False, name=None))
surv_models = list(ss.loc[~ss["overfolded"], ["cluster", "fname"]].itertuples(index=False, name=None))
print(f"all energy-filtered: {len(ef_models)} models   |   over-folding survivors: {len(surv_models)} models\n")

pose_all  = pose_pipeline(ef_models,   "all energy-filtered")
pose_surv = pose_pipeline(surv_models, "survivors only")

all energy-filtered: 455 models   |   over-folding survivors: 194 models



[all energy-filtered] n=455  DRB2 core=156  DRB4 trim kept=142  best_k=2 (silhouette 0.359)  PCA var=[np.float32(0.412), np.float32(0.177)]  sizes={2: 451, 1: 4}


[survivors only] n=194  DRB2 core=156  DRB4 trim kept=142  best_k=2 (silhouette 0.451)  PCA var=[np.float32(0.533), np.float32(0.157)]  sizes={2: 162, 1: 32}


### PCA scatter, silhouette curve, and cluster composition — before vs after

In [19]:
# --- PCA scatter (re-derived), coloured by re-derived pose cluster ---------
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"all energy-filtered (n={len(pose_all['df'])})",
    f"over-folding survivors (n={len(pose_surv['df'])})"])
for j, res in enumerate([pose_all, pose_surv], start=1):
    d = res["df"]
    for lab in sorted(d["pose_label"].unique()):
        sub = d[d["pose_label"] == lab]
        fig.add_trace(go.Scatter(
            x=sub["pc1"], y=sub["pc2"], mode="markers",
            name=f"{res['tag']} · pose {lab} (n={len(sub)})",
            marker=dict(size=6, opacity=0.75),
            customdata=sub[["backend", "fname"]],
            hovertemplate="pose %{fullData.name}<br>%{customdata[0]}<br>%{customdata[1]}"
                          "<br>pc1=%{x:.1f} pc2=%{y:.1f}<extra></extra>"),
            row=1, col=j)
    fig.update_xaxes(title_text="PC1", row=1, col=j)
    fig.update_yaxes(title_text="PC2", row=1, col=j)
fig.update_layout(title="Re-derived pose PCA — coloured by re-derived hierarchical cluster",
                  template=TEMPLATE, width=1100, height=520,
                  legend=dict(orientation="h", yanchor="bottom", y=-0.28))
save_fig(fig, "pose_pca_filtered_vs_all.html", "pose_reclustering")

# --- silhouette vs k ------------------------------------------------------
sil_df = pd.concat([
    pd.DataFrame({"k": list(pose_all["sil"]),  "silhouette": list(pose_all["sil"].values()),  "set": "all energy-filtered"}),
    pd.DataFrame({"k": list(pose_surv["sil"]), "silhouette": list(pose_surv["sil"].values()), "set": "survivors only"}),
])
figs = px.line(sil_df, x="k", y="silhouette", color="set", markers=True, template=TEMPLATE,
               title="Hierarchical-clustering silhouette vs k — full set vs filtered subset")
figs.update_layout(width=760, height=420)
save_fig(figs, "pose_silhouette_vs_k.html", "pose_reclustering")

# --- cluster composition by backend ------------------------------------
for res in (pose_all, pose_surv):
    print(f"\n[{res['tag']}] re-derived pose cluster x backend:")
    display(pd.crosstab(res["df"]["backend"], res["df"]["pose_label"], margins=True))

# --- does the filtered 2-cluster split line up with the over-folding
#     survivor / dropped partition, or with backend? -----------------------
comp = pose_surv["df"].merge(ss[["fname", "cluster", "backend", "max_run", "worst_pct"]],
                             on=["fname", "cluster", "backend"], how="left")
print("\n[survivors] median non-MoRF helix stats per re-derived pose cluster:")
display(comp.groupby("pose_label")[["worst_pct", "max_run"]].median().round(1))

Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/pose_reclustering/pose_pca_filtered_vs_all.html


Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/pose_reclustering/pose_silhouette_vs_k.html



[all energy-filtered] re-derived pose cluster x backend:


pose_label,1,2,All
backend,,,
alphafold3,0,99,99
boltz,0,99,99
chai1,0,99,99
openfold3,4,65,69
protenix,0,89,89
All,4,451,455



[survivors only] re-derived pose cluster x backend:


pose_label,1,2,All
backend,,,
alphafold3,5,77,82
boltz,5,0,5
chai1,15,35,50
protenix,7,50,57
All,32,162,194



[survivors] median non-MoRF helix stats per re-derived pose cluster:


,worst_pct,max_run
pose_label,,
1,14.9,7.0
2,8.8,6.0


### Sample 20 structures per re-derived pose cluster for ChimeraX

Random 20 (or all, if fewer) models per re-derived pose cluster, symlinked
into `results/rna_ds_drb2_drb4/pose_reclustering_samples/<set>/pose_<label>/`
so a manageable subset can be opened side by side in ChimeraX. `<set>` is
`survivors` (the 162/32 split — the interesting one) and `all` (the full
energy-filtered set, whose "pose 1" is just the handful of outliers).

In [20]:
import shutil

SAMPLE_N = 20
_RNG_SEED = 0

def _sample_symlinks(pose_df, set_tag):
    root = RESULTS_DIR / "pose_reclustering_samples" / set_tag
    if root.exists():
        shutil.rmtree(root)
    root.mkdir(parents=True)
    mroot = (RESULTS_DIR / "minimized").resolve()
    print(f"[{set_tag}] -> {root}")
    for lab in sorted(pose_df["pose_label"].unique()):
        g = pose_df[pose_df["pose_label"] == lab]
        picks = g.sample(n=min(SAMPLE_N, len(g)), random_state=_RNG_SEED)
        d = root / f"pose_{lab}"
        d.mkdir(parents=True, exist_ok=True)
        n = 0
        for row in picks.itertuples(index=False):
            src = mroot / str(row.cluster) / row.fname / f"{row.fname}.pdb"
            if not src.exists():
                continue
            (d / f"pose{lab}_{row.backend}_{row.fname}.pdb").symlink_to(src)
            n += 1
        print(f"  pose {lab}: {n} / {len(g)} models sampled  "
              f"({g['backend'].value_counts().to_dict()})")
    print(f"Done: {root}\n")

_sample_symlinks(pose_surv["df"], "survivors")
_sample_symlinks(pose_all["df"],  "all")


[survivors] -> ../results/rna_ds_drb2_drb4/pose_reclustering_samples/survivors
  pose 1: 20 / 32 models sampled  ({'chai1': 15, 'protenix': 7, 'alphafold3': 5, 'boltz': 5})
  pose 2: 20 / 162 models sampled  ({'alphafold3': 77, 'protenix': 50, 'chai1': 35})
Done: ../results/rna_ds_drb2_drb4/pose_reclustering_samples/survivors

[all] -> ../results/rna_ds_drb2_drb4/pose_reclustering_samples/all
  pose 1: 4 / 4 models sampled  ({'openfold3': 4})
  pose 2: 20 / 451 models sampled  ({'boltz': 99, 'alphafold3': 99, 'chai1': 99, 'protenix': 89, 'openfold3': 65})
Done: ../results/rna_ds_drb2_drb4/pose_reclustering_samples/all



### Reading this — yes, filtering unmasks a second pose

|  | all energy-filtered (n=455) | survivors only (n=194) |
|---|---|---|
| best k (silhouette) | 2 | 2 |
| silhouette @ k=2 | 0.359 | **0.451** |
| cluster sizes | **451 / 4** (one pose + 4 outliers) | **162 / 32** (~84 / 16 %) |
| PC1 explained variance | 0.41 | **0.53** |
| DRB2 core / DRB4 trim | 156 / 142 | 156 / 142 (identical) |

- **Full set:** one pose plus a 4-model outlier blob (all OpenFold3) — the
  intro's "599/600 in one cluster" reproduced.
- **Survivors:** the same `k=2` now splits **162 / 32** — a real minority
  pose, ~16 % of the surviving ensemble, that the over-folded models had been
  smearing into noise. The silhouette lifts (0.36 → 0.45) and PC1 alone now
  carries 53 % of the pose variance vs 41 %. The DRB2 rigid core and the DRB4
  RMSF-trimmed footprint come out identical (156 / 142 Cα), so this is a
  genuine change in the *pose distribution*, not a change in what was measured.
- **The minority pose is architecture-mixed, not a single-backend artefact:**
  by backend it is AF3 5 / Boltz 5 / Chai-1 15 / Protenix 7 — every one of the
  5 surviving Boltz models sits in it and Chai-1 is over-represented (15 / 50),
  but AF3 (5 / 82) and Protenix (7 / 57) contribute a handful too. Its models
  carry slightly more residual non-MoRF helix (median worst-domain 14.9 % vs
  8.8 %), consistent with the Boltz/Chai-1 lean.

**Takeaway:** the pipeline's "single converged pose" for this complex is partly
an artefact of the disordered-region over-folding drowning out a second
DRB2–DRB4 arrangement. Once the over-folded models are removed, ~1 in 6
survivors adopt a distinct pose worth inspecting (the `overfolding_inspection_survivors/`
symlinks + the re-derived `pose_label` in `pose_surv["df"]` identify them).
Worth re-running the real `scripts/pose_cluster_anchor.py` on the survivor
subset (raw CIFs) to confirm.

**Caveat on the exact split size:** like `pose_cluster_anchor.py`, this uses the first model as the Kabsch alignment reference, so the minority-pose count is somewhat reference-dependent (a different first model gave ~137/57 in a separate run). What is stable is the *direction*: the full set has one pose + ~4 outliers, the survivor set has a substantial second group (16-30 %). Re-run the real script on the survivor CIFs to pin it down.